# Gold Analytics — Business Intelligence

## Notebook Overview

### Purpose
Perform business-oriented analytical queries on the Gold dimensional model using Spark SQL. The objective is to transform curated Gold data into actionable business insights.

### Source Tables
- `gold_fact_sales`
- `gold_dim_customer`
- `gold_dim_product`
- `gold_dim_date`

### Analytical Areas
1. Overall sales performance
2. Monthly and yearly sales trends
3. Product and category performance
4. Customer and customer-segment analysis
5. Order status analysis
6. Payment status analysis

### Configuration / Paths

In [1]:
from google.colab import drive
drive.mount("/content/drive")

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"

GOLD_FACT_SALES_PATH = f"{FILE_PATH}/gold/fact_sales"
GOLD_CUSTOMER_PATH = f"{FILE_PATH}/gold/dim_customer"
GOLD_PRODUCT_PATH = f"{FILE_PATH}/gold/dim_product"
GOLD_DATE_PATH = f"{FILE_PATH}/gold/dim_date"

Mounted at /content/drive


### Spark Session and Load Gold Tables

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("gold-analytics")
    .getOrCreate()
)

df_fact_sales = spark.read.parquet(GOLD_FACT_SALES_PATH)

df_dim_customer = spark.read.parquet(GOLD_CUSTOMER_PATH)
df_dim_product = spark.read.parquet(GOLD_PRODUCT_PATH)
df_dim_date = spark.read.parquet(GOLD_DATE_PATH)

print("Gold fact rows:", df_fact_sales.count())
print("Customer dimension rows:", df_dim_customer.count())
print("Product dimension rows:", df_dim_product.count())
print("Date dimension rows:", df_dim_date.count())

Gold fact rows: 49990
Customer dimension rows: 9997
Product dimension rows: 996
Date dimension rows: 731


### Register Temporary SQL Views

In [3]:
df_fact_sales.createOrReplaceTempView("gold_fact_sales")
df_dim_customer.createOrReplaceTempView("gold_dim_customer")
df_dim_product.createOrReplaceTempView("gold_dim_product")
df_dim_date.createOrReplaceTempView("gold_dim_date")

print("Gold SQL views registered successfully.")

Gold SQL views registered successfully.


### Inspect the Final Fact Schema

In [4]:
df_fact_sales.printSchema()

print("Available fact columns:")
print(df_fact_sales.columns)

df_fact_sales.show(10, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_amount: decimal(10,2) (nullable = true)
 |-- tax_amount: decimal(10,2) (nullable = true)
 |-- transaction_amount: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _gold_processed_timestamp: timestamp (nullable = true)

Available fact columns:
['transaction_id', 'order_id', 'customer_id', 'product_id', 'date_key', 'order_date', 'order_status', 'payment_status', 'quantity', 'unit_price', 'discount_amount', 'tax_amount', 'transaction_amount', '_silver_processed_timestamp', '_gold_processed_timesta

## ***Overall Sales Performance***

### Business Objective
Evaluate the overall performance of the sales transactions by calculating key financial and operational metrics.

### Key Metrics
- Total Transactions
- Total Recorded Transaction Amount
- Total Quantity Sold
- Average Transaction Amount
- Total Discount Amount
- Total Tax Amount
- Average Unit Price

### Analytical Approach
Use aggregate SQL functions on `gold_fact_sales` to calculate the overall metrics at transaction grain.

In [5]:
df_overall_sales = spark.sql("""
    SELECT
        COUNT(DISTINCT transaction_id) AS total_transactions,
        SUM(transaction_amount) AS total_transaction_amount,
        SUM(quantity) AS total_quantity_sold,

        ROUND(
            SUM(transaction_amount) / COUNT(DISTINCT transaction_id),
            2
        ) AS average_transaction_amount,

        SUM(discount_amount) AS total_discount_amount,
        SUM(tax_amount) AS total_tax_amount,
        ROUND(AVG(unit_price), 2) AS average_unit_price

    FROM gold_fact_sales
""")

df_overall_sales.show(truncate=False)

+------------------+------------------------+-------------------+--------------------------+---------------------+----------------+------------------+
|total_transactions|total_transaction_amount|total_quantity_sold|average_transaction_amount|total_discount_amount|total_tax_amount|average_unit_price|
+------------------+------------------------+-------------------+--------------------------+---------------------+----------------+------------------+
|49990             |23501845.78             |149852             |470.13                    |1133229.16           |1740878.42      |152.95            |
+------------------+------------------------+-------------------+--------------------------+---------------------+----------------+------------------+



### Validate the Result

In [6]:
source_transaction_count = df_fact_sales.count()

kpi_transaction_count = (
    df_overall_sales
    .select("total_transactions")
    .first()["total_transactions"]
)

print("Fact table transaction rows:", source_transaction_count)
print("Distinct transactions in KPI:", kpi_transaction_count)

assert source_transaction_count == kpi_transaction_count, (
    "Transaction count mismatch: check for duplicate transaction IDs."
)

print("Overall sales KPI validation passed.")

Fact table transaction rows: 49990
Distinct transactions in KPI: 49990
Overall sales KPI validation passed.


### Key Findings — Overall Sales Performance

- *Total recorded transactions*: **49,990**
- *Total recorded transaction amount*: **23,501,845.78**
- *Total quantity sold*: **149,852**
- *Average transaction amount*: **470.13**
- *Total discounts recorded*: **1,133,229.16**
- *Total taxes recorded*: **1,740,878.42**
- *Average unit price*: **152.95**

The dataset provides a baseline for evaluating transaction performance, pricing, discounts, and taxes. Further analysis will investigate how these metrics vary across time, customers, and products.

***Note:*** Transaction amount is the recorded transaction value and should not automatically be interpreted as recognized revenue or profit.

## ***Monthly Sales Trend Analysis***

### Business Objective
Understand monthly transaction performance and identify changes in transaction volume, recorded transaction amounts, and quantity sold over time.

### Dimensions Used
- `gold_fact_sales`
- `gold_dim_date`

### Metrics
- Monthly transaction count
- Monthly recorded transaction amount
- Monthly quantity sold
- Monthly average transaction amount

### Analytical Approach
Join the date dimension with the sales fact using `date_key`, then aggregate transaction measures by calendar year and month.

In [7]:
df_monthly_sales = spark.sql("""
    SELECT
        d.year,
        d.month,
        d.month_name,

        COUNT(DISTINCT f.transaction_id) AS total_transactions,

        COALESCE(
            SUM(f.transaction_amount), 0
        ) AS total_transaction_amount,

        COALESCE(
            SUM(f.quantity), 0
        ) AS total_quantity_sold,

        ROUND(
            SUM(f.transaction_amount) /
            NULLIF(COUNT(DISTINCT f.transaction_id), 0),
            2
        ) AS average_transaction_amount

    FROM gold_dim_date d

    LEFT JOIN gold_fact_sales f
        ON d.date_key = f.date_key

    GROUP BY
        d.year,
        d.month,
        d.month_name

    ORDER BY
        d.year,
        d.month
""")

df_monthly_sales.show(100, truncate=False)

+----+-----+----------+------------------+------------------------+-------------------+--------------------------+
|year|month|month_name|total_transactions|total_transaction_amount|total_quantity_sold|average_transaction_amount|
+----+-----+----------+------------------+------------------------+-------------------+--------------------------+
|2024|1    |January   |2111              |995862.07               |6402               |471.75                    |
|2024|2    |February  |1980              |906966.34               |5939               |458.06                    |
|2024|3    |March     |2064              |980586.07               |6237               |475.09                    |
|2024|4    |April     |2015              |968623.29               |6062               |480.71                    |
|2024|5    |May       |2113              |991927.89               |6384               |469.44                    |
|2024|6    |June      |2062              |942177.42               |6106         

### *Best-performing month*

In [8]:
df_monthly_sales.orderBy(
    F.col("total_transaction_amount").desc()
).show(1, truncate=False)

+----+-----+----------+------------------+------------------------+-------------------+--------------------------+
|year|month|month_name|total_transactions|total_transaction_amount|total_quantity_sold|average_transaction_amount|
+----+-----+----------+------------------+------------------------+-------------------+--------------------------+
|2025|8    |August    |2197              |1049730.00              |6569               |477.80                    |
+----+-----+----------+------------------+------------------------+-------------------+--------------------------+
only showing top 1 row


### Validate monthly aggregation

In [9]:
monthly_total = (
    df_monthly_sales
    .agg(F.sum("total_transaction_amount").alias("monthly_total"))
    .first()["monthly_total"]
)

matched_date_total = (
    spark.sql("""
        SELECT SUM(f.transaction_amount) AS matched_date_total
        FROM gold_fact_sales f
        INNER JOIN gold_dim_date d
            ON f.date_key = d.date_key
    """)
    .first()["matched_date_total"]
)

print("Monthly aggregated amount:", monthly_total)
print("Fact amount with matching dates:", matched_date_total)

assert round(float(monthly_total or 0), 2) == round(
    float(matched_date_total or 0), 2
)

print("Monthly aggregation validation passed.")

Monthly aggregated amount: 23501845.78
Fact amount with matching dates: 23501845.78
Monthly aggregation validation passed.


## ***Product & Category Performance***

### Objective
Analyze product and category performance using the Gold fact and product dimension tables.

### Key metrics
- Total transactions
- Total quantity sold
- Total transaction amount
- Average transaction amount

### Business questions
- Which product categories generate the highest transaction amount?
- Which categories contribute the most sales volume?
- Are high-volume categories also high-value categories?

### Category Level Performance

In [10]:
df_category_performance = spark.sql("""
    SELECT
        p.category,
        COUNT(DISTINCT f.transaction_id) AS total_transactions,
        COALESCE(SUM(f.quantity), 0) AS total_quantity_sold,
        COALESCE(SUM(f.transaction_amount), 0) AS total_transaction_amount,
        ROUND(
            SUM(f.transaction_amount) /
            NULLIF(COUNT(DISTINCT f.transaction_id), 0),
            2
        ) AS average_transaction_amount
    FROM gold_dim_product p
    LEFT JOIN gold_fact_sales f
        ON p.product_id = f.product_id
    GROUP BY p.category
    ORDER BY total_transaction_amount DESC
""")

df_category_performance.show(100, truncate=False)

+--------------+------------------+-------------------+------------------------+--------------------------+
|category      |total_transactions|total_quantity_sold|total_transaction_amount|average_transaction_amount|
+--------------+------------------+-------------------+------------------------+--------------------------+
|Home & Kitchen|10505             |31133              |4899779.98              |466.42                    |
|Sports        |10193             |30553              |4782570.58              |469.20                    |
|Beauty        |10051             |30220              |4739912.61              |471.59                    |
|Books         |9714              |29351              |4612037.28              |474.78                    |
|Electronics   |9344              |28042              |4381417.02              |468.90                    |
+--------------+------------------+-------------------+------------------------+--------------------------+



### Product Level Performance

In [11]:
df_product_performance = spark.sql("""
    SELECT
        p.product_id,
        p.product_name,
        p.brand,
        p.category,
        p.subcategory,
        COUNT(DISTINCT f.transaction_id) AS total_transactions,
        COALESCE(SUM(f.quantity), 0) AS total_quantity_sold,
        COALESCE(SUM(f.transaction_amount), 0) AS total_transaction_amount
    FROM gold_dim_product p
    LEFT JOIN gold_fact_sales f
        ON p.product_id = f.product_id
    GROUP BY
        p.product_id,
        p.product_name,
        p.brand,
        p.category,
        p.subcategory
    ORDER BY total_transaction_amount DESC
""")

df_product_performance.show(20, truncate=False)

+----------+----------------------+------+--------------+-----------+------------------+-------------------+------------------------+
|product_id|product_name          |brand |category      |subcategory|total_transactions|total_quantity_sold|total_transaction_amount|
+----------+----------------------+------+--------------+-----------+------------------+-------------------+------------------------+
|P00105    |Orion Wellness 0105   |Cedar |Beauty        |Wellness   |75                |233                |40761.51                |
|P00997    |Harbor Phones 0997    |Atlas |Electronics   |Phones     |67                |210                |39192.16                |
|P00360    |Cedar Technology 0360 |Peak  |Books         |Technology |66                |232                |38251.98                |
|P00050    |Luma Makeup 0050      |Cedar |Beauty        |Makeup     |68                |214                |38078.21                |
|P00281    |Orion Makeup 0281     |Atlas |Beauty        |Makeu

### Validation

In [12]:
category_total = (
    df_category_performance
    .agg(F.sum("total_transaction_amount").alias("total"))
    .first()["total"]
)

product_total = (
    df_product_performance
    .agg(F.sum("total_transaction_amount").alias("total"))
    .first()["total"]
)

print("Category total:", category_total)
print("Product total:", product_total)

assert round(float(category_total or 0), 2) == round(
    float(product_total or 0), 2
)

print("Validation passed!")

Category total: 23415717.47
Product total: 23415717.47
Validation passed!


## ***Customer Performance & Segmentation***

### Objective
Analyze customer purchasing behavior and contribution using the Gold customer dimension and sales fact.

### Business questions
- Which customer segments contribute the most transaction amount?
- Which countries generate the highest transaction amount?
- Who are the top 10 customers by transaction amount?
- How does customer transaction activity vary across segments?

### Key metrics
- Total customers
- Total transactions
- Total quantity sold
- Total transaction amount
- Average transaction amount per transaction

### Customer-Segment Level

In [13]:
df_segment_performance = spark.sql("""
    SELECT
        c.customer_segment,
        COUNT(DISTINCT c.customer_id) AS total_customers,
        COUNT(DISTINCT f.transaction_id) AS total_transactions,
        COALESCE(SUM(f.quantity), 0) AS total_quantity_sold,
        COALESCE(SUM(f.transaction_amount), 0) AS total_transaction_amount,
        ROUND(
            SUM(f.transaction_amount) /
            NULLIF(COUNT(DISTINCT f.transaction_id), 0),
            2
        ) AS average_transaction_amount
    FROM gold_dim_customer c
    LEFT JOIN gold_fact_sales f
        ON c.customer_id = f.customer_id
    GROUP BY c.customer_segment
    ORDER BY total_transaction_amount DESC
""")

df_segment_performance.show(100, truncate=False)

+----------------+---------------+------------------+-------------------+------------------------+--------------------------+
|customer_segment|total_customers|total_transactions|total_quantity_sold|total_transaction_amount|average_transaction_amount|
+----------------+---------------+------------------+-------------------+------------------------+--------------------------+
|Consumer        |3369           |16942             |50662              |7934808.53              |468.35                    |
|Enterprise      |3285           |16587             |49593              |7790471.51              |469.67                    |
|Small Business  |3342           |16445             |49548              |7769333.22              |472.44                    |
|Premier         |1              |0                 |0                  |0.00                    |NULL                      |
+----------------+---------------+------------------+-------------------+------------------------+--------------------

### Country-level customer performance

In [14]:
df_country_performance = spark.sql("""
    SELECT
        c.country,
        COUNT(DISTINCT c.customer_id) AS total_customers,
        COUNT(DISTINCT f.transaction_id) AS total_transactions,
        COALESCE(SUM(f.quantity), 0) AS total_quantity_sold,
        COALESCE(SUM(f.transaction_amount), 0) AS total_transaction_amount
    FROM gold_dim_customer c
    LEFT JOIN gold_fact_sales f
        ON c.customer_id = f.customer_id
    GROUP BY c.country
    ORDER BY total_transaction_amount DESC
""")

df_country_performance.show(100, truncate=False)

+--------------+---------------+------------------+-------------------+------------------------+
|country       |total_customers|total_transactions|total_quantity_sold|total_transaction_amount|
+--------------+---------------+------------------+-------------------+------------------------+
|United States |1736           |8868              |26636              |4196630.50              |
|Canada        |1714           |8549              |25569              |4000086.63              |
|Australia     |1669           |8451              |25421              |3985713.73              |
|United Kingdom|1621           |8185              |24498              |3890507.10              |
|Germany       |1619           |8077              |24250              |3758363.28              |
|India         |1638           |7844              |23429              |3663312.02              |
+--------------+---------------+------------------+-------------------+------------------------+



### Top 10 customers

In [15]:
df_top_customers = spark.sql("""
    SELECT
        c.customer_id,
        CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
        c.customer_segment,
        c.country,
        COUNT(DISTINCT f.transaction_id) AS total_transactions,
        COALESCE(SUM(f.quantity), 0) AS total_quantity_sold,
        COALESCE(SUM(f.transaction_amount), 0) AS total_transaction_amount,
        ROUND(
            SUM(f.transaction_amount) /
            NULLIF(COUNT(DISTINCT f.transaction_id), 0),
            2
        ) AS average_transaction_amount
    FROM gold_dim_customer c
    LEFT JOIN gold_fact_sales f
        ON c.customer_id = f.customer_id
    GROUP BY
        c.customer_id,
        c.first_name,
        c.last_name,
        c.customer_segment,
        c.country
    ORDER BY total_transaction_amount DESC
    LIMIT 10
""")

df_top_customers.show(10, truncate=False)

+-----------+-------------+----------------+--------------+------------------+-------------------+------------------------+--------------------------+
|customer_id|customer_name|customer_segment|country       |total_transactions|total_quantity_sold|total_transaction_amount|average_transaction_amount|
+-----------+-------------+----------------+--------------+------------------+-------------------+------------------------+--------------------------+
|C008617    |Isha Smith   |Enterprise      |Germany       |19                |63                 |11930.08                |627.90                    |
|C000202    |Emma Gupta   |Small Business  |Australia     |21                |67                 |11785.18                |561.20                    |
|C008465    |Mia Gupta    |Small Business  |Germany       |20                |66                 |11391.71                |569.59                    |
|C003810    |Emma Davis   |Small Business  |Germany       |18                |54              

### Data reconciliation

In [16]:
customer_matched_total = (
    df_segment_performance
    .agg(F.sum("total_transaction_amount").alias("total"))
    .first()["total"]
)

fact_matched_total = spark.sql("""
    SELECT SUM(f.transaction_amount) AS total
    FROM gold_fact_sales f
    INNER JOIN gold_dim_customer c
        ON f.customer_id = c.customer_id
""").first()["total"]

print("Customer aggregation total:", customer_matched_total)
print("Matched fact total:", fact_matched_total)

assert round(float(customer_matched_total or 0), 2) == round(
    float(fact_matched_total or 0), 2
)

print("Customer reconciliation passed!")

Customer aggregation total: 23494613.26
Matched fact total: 23494613.26
Customer reconciliation passed!


## ***Payment & Order Status Analysis***

### Objective
Analyze payment outcomes and order status distribution using the Gold sales fact table.

### Business questions
- What is the distribution of payment statuses?
- How much transaction value is associated with each payment status?
- What is the distribution of order statuses?
- Are there differences between payment status and order status?

### Key metrics
- Total transactions
- Transaction count by status
- Transaction amount by status
- Transaction percentage by status

### Distribution of Payment Analysis


In [17]:
df_payment_status = spark.sql("""
    SELECT
        COALESCE(payment_status, 'UNKNOWN') AS payment_status,
        COUNT(DISTINCT transaction_id) AS total_transactions,
        COALESCE(SUM(transaction_amount), 0) AS total_transaction_amount,
        COALESCE(SUM(quantity), 0) AS total_quantity_sold,
        ROUND(
            COUNT(DISTINCT transaction_id) * 100.0 /
            SUM(COUNT(DISTINCT transaction_id)) OVER (),
            2
        ) AS transaction_percentage
    FROM gold_fact_sales
    GROUP BY COALESCE(payment_status, 'UNKNOWN')
    ORDER BY total_transactions DESC
""")

df_payment_status.show(100, truncate=False)

+--------------+------------------+------------------------+-------------------+----------------------+
|payment_status|total_transactions|total_transaction_amount|total_quantity_sold|transaction_percentage|
+--------------+------------------+------------------------+-------------------+----------------------+
|Captured      |39091             |18391617.67             |117124             |78.20                 |
|Refunded      |4226              |1966079.56              |12569              |8.45                  |
|Pending       |3878              |1816320.66              |11738              |7.76                  |
|Failed        |2794              |1326321.77              |8416               |5.59                  |
|Settled       |1                 |1506.12                 |5                  |0.00                  |
+--------------+------------------+------------------------+-------------------+----------------------+



### Order Status Analysis

In [18]:
df_order_status = spark.sql("""
    SELECT
        COALESCE(order_status, 'UNKNOWN') AS order_status,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT transaction_id) AS total_transactions,
        COALESCE(SUM(transaction_amount), 0) AS total_transaction_amount,
        COALESCE(SUM(quantity), 0) AS total_quantity_sold,
        ROUND(
            COUNT(DISTINCT order_id) * 100.0 /
            SUM(COUNT(DISTINCT order_id)) OVER (),
            2
        ) AS order_percentage
    FROM gold_fact_sales
    GROUP BY COALESCE(order_status, 'UNKNOWN')
    ORDER BY total_orders DESC
""")

df_order_status.show(100, truncate=False)

+------------+------------+------------------+------------------------+-------------------+----------------+
|order_status|total_orders|total_transactions|total_transaction_amount|total_quantity_sold|order_percentage|
+------------+------------+------------------+------------------------+-------------------+----------------+
|Completed   |15236       |31149             |14677737.01             |93531              |62.51           |
|Shipped     |3889        |8012              |3731612.38              |23849              |15.96           |
|Processing  |2371        |4854              |2284745.53              |14596              |9.73            |
|Cancelled   |1715        |3591              |1702235.27              |10801              |7.04            |
|Returned    |1159        |2379              |1103636.22              |7061               |4.76            |
|UNKNOWN     |2           |4                 |1248.87                 |9                  |0.01            |
|In Transit  |1    

### Payment status vs order status

In [19]:
df_payment_order_matrix = spark.sql("""
    SELECT
        COALESCE(payment_status, 'UNKNOWN') AS payment_status,
        COALESCE(order_status, 'UNKNOWN') AS order_status,
        COUNT(DISTINCT transaction_id) AS total_transactions,
        COUNT(DISTINCT order_id) AS total_orders,
        COALESCE(SUM(transaction_amount), 0) AS total_transaction_amount
    FROM gold_fact_sales
    GROUP BY
        COALESCE(payment_status, 'UNKNOWN'),
        COALESCE(order_status, 'UNKNOWN')
    ORDER BY
        total_transactions DESC
""")

df_payment_order_matrix.show(100, truncate=False)

+--------------+------------+------------------+------------+------------------------+
|payment_status|order_status|total_transactions|total_orders|total_transaction_amount|
+--------------+------------+------------------+------------+------------------------+
|Captured      |Completed   |25500             |13993       |12023519.92             |
|Captured      |Shipped     |6560              |3556        |3030597.18              |
|Captured      |Processing  |3997              |2172        |1904704.85              |
|Captured      |Cancelled   |3029              |1598        |1430916.35              |
|Pending       |Completed   |2581              |2409        |1209621.76              |
|Refunded      |Returned    |2379              |1159        |1103636.22              |
|Failed        |Completed   |1851              |1761        |871925.22               |
|Refunded      |Completed   |1216              |1184        |571163.99               |
|Pending       |Shipped     |645           

### Payment and order status reconciliation

In [21]:
payment_total = (
    df_payment_status
    .agg(F.sum("total_transactions").alias("total"))
    .first()["total"]
)

fact_total = df_fact_sales.select(
    F.countDistinct("transaction_id").alias("total")
).first()["total"]

print("Payment status transaction count:", payment_total)
print("Fact table transaction count:", fact_total)

assert payment_total == fact_total

print("Payment status reconciliation passed!")

order_status_total = (
    df_order_status
    .agg(F.sum("total_orders").alias("total"))
    .first()["total"]
)

fact_distinct_orders = df_fact_sales.select(
    F.countDistinct("order_id").alias("total")
).first()["total"]

print("Order status distinct order count:", order_status_total)
print("Fact table distinct order count:", fact_distinct_orders)

assert order_status_total == fact_distinct_orders

print("Order status reconciliation passed!")

Payment status transaction count: 49990
Fact table transaction count: 49990
Payment status reconciliation passed!
Order status distinct order count: 24373
Fact table distinct order count: 24373
Order status reconciliation passed!


### Inconsistent Order Statuses

In [22]:
df_inconsistent_order_status = spark.sql("""
    SELECT
        order_id,
        COUNT(DISTINCT order_status) AS distinct_status_count,
        COLLECT_SET(order_status) AS order_statuses
    FROM gold_fact_sales
    WHERE order_id IS NOT NULL
    GROUP BY order_id
    HAVING COUNT(DISTINCT order_status) > 1
    ORDER BY distinct_status_count DESC
""")

df_inconsistent_order_status.show(100, truncate=False)

+--------+---------------------+--------------+
|order_id|distinct_status_count|order_statuses|
+--------+---------------------+--------------+
+--------+---------------------+--------------+



## *Discount & Tax Analysis*

### Objective
Analyze discount and tax amounts recorded in the Gold sales fact table to understand their contribution to transaction values and their distribution across transactions.

### Business questions
- What is the total discount amount recorded across all transactions?
- What is the average discount amount per transaction?
- How much tax is recorded across all transactions?
- What is the average tax amount per transaction?
- What percentage of the pre-discount unit-price value is represented by discounts?
- How does the discount and tax contribution vary across payment statuses?

### Key metrics
- Total discount amount
- Average discount amount
- Total tax amount
- Average tax amount
- Total transaction amount
- Discount percentage
- Tax percentage
- Discount and tax amount by payment status

### Overall Discount and Tax Summary

In [23]:
df_discount_tax_summary = spark.sql("""
    SELECT
        COUNT(DISTINCT transaction_id) AS total_transactions,

        ROUND(SUM(discount_amount), 2) AS total_discount_amount,

        ROUND(AVG(discount_amount), 2) AS average_discount_amount,

        ROUND(SUM(tax_amount), 2) AS total_tax_amount,

        ROUND(AVG(tax_amount), 2) AS average_tax_amount,

        ROUND(SUM(transaction_amount), 2) AS total_transaction_amount,

        ROUND(
            SUM(discount_amount) * 100.0 /
            NULLIF(SUM(unit_price * quantity), 0),
            2
        ) AS discount_percentage,

        ROUND(
            SUM(tax_amount) * 100.0 /
            NULLIF(SUM(transaction_amount), 0),
            2
        ) AS tax_percentage

    FROM gold_fact_sales
""")

df_discount_tax_summary.show(truncate=False)

+------------------+---------------------+-----------------------+----------------+------------------+------------------------+-------------------+--------------+
|total_transactions|total_discount_amount|average_discount_amount|total_tax_amount|average_tax_amount|total_transaction_amount|discount_percentage|tax_percentage|
+------------------+---------------------+-----------------------+----------------+------------------+------------------------+-------------------+--------------+
|49990             |1133229.16           |22.67                  |1740878.42      |34.82             |23501845.78             |4.95               |7.41          |
+------------------+---------------------+-----------------------+----------------+------------------+------------------------+-------------------+--------------+



### Discount Distribution



In [25]:
df_discount_distribution = spark.sql("""
    SELECT
        COUNT(DISTINCT transaction_id) AS total_transactions,

        COUNT(DISTINCT CASE
            WHEN discount_amount > 0 THEN transaction_id
        END) AS discounted_transactions,

        COUNT(DISTINCT CASE
            WHEN discount_amount = 0 THEN transaction_id
        END) AS non_discounted_transactions,

        COUNT(DISTINCT CASE
            WHEN discount_amount IS NULL THEN transaction_id
        END) AS transactions_with_null_discount,

        ROUND(
            COUNT(DISTINCT CASE
                WHEN discount_amount > 0 THEN transaction_id
            END) * 100.0 /
            COUNT(DISTINCT transaction_id),
            2
        ) AS discounted_transaction_percentage,

        ROUND(AVG(
            CASE WHEN discount_amount > 0
                 THEN discount_amount
            END
        ), 2) AS average_discount_amount,

        ROUND(MAX(discount_amount), 2) AS maximum_discount_amount

    FROM gold_fact_sales
""")

df_discount_distribution.show(truncate=False)

+------------------+-----------------------+---------------------------+-------------------------------+---------------------------------+-----------------------+-----------------------+
|total_transactions|discounted_transactions|non_discounted_transactions|transactions_with_null_discount|discounted_transaction_percentage|average_discount_amount|maximum_discount_amount|
+------------------+-----------------------+---------------------------+-------------------------------+---------------------------------+-----------------------+-----------------------+
|49990             |24938                  |25052                      |0                              |49.89                            |45.44                  |224.99                 |
+------------------+-----------------------+---------------------------+-------------------------------+---------------------------------+-----------------------+-----------------------+



### Validation

In [26]:
df_discount_distribution_validation = spark.sql("""
    SELECT
        COUNT(*) AS total_fact_rows,
        COUNT(DISTINCT transaction_id) AS distinct_transactions,
        COUNT(DISTINCT CASE
            WHEN discount_amount > 0 THEN transaction_id
        END) AS discounted_transactions,
        COUNT(DISTINCT CASE
            WHEN discount_amount = 0 THEN transaction_id
        END) AS non_discounted_transactions,
        COUNT(DISTINCT CASE
            WHEN discount_amount IS NULL THEN transaction_id
        END) AS null_discount_transactions
    FROM gold_fact_sales
""")

df_discount_distribution_validation.show(truncate=False)

+---------------+---------------------+-----------------------+---------------------------+--------------------------+
|total_fact_rows|distinct_transactions|discounted_transactions|non_discounted_transactions|null_discount_transactions|
+---------------+---------------------+-----------------------+---------------------------+--------------------------+
|49990          |49990                |24938                  |25052                      |0                         |
+---------------+---------------------+-----------------------+---------------------------+--------------------------+



### Top 10 transactions by discount

In [28]:
df_top_discount_transactions = spark.sql("""
    SELECT
        transaction_id,
        order_id,
        product_id,
        quantity,
        unit_price,
        discount_amount,
        ROUND(unit_price * quantity, 2) AS pre_discount_value,
        ROUND(
            discount_amount * 100.0 /
            NULLIF(unit_price * quantity, 0),
            2
        ) AS discount_percentage

    FROM gold_fact_sales
    WHERE discount_amount > 0
    ORDER BY discount_amount DESC
    LIMIT 10
""")

df_top_discount_transactions.show(truncate=False)

+--------------+--------+----------+--------+----------+---------------+------------------+-------------------+
|transaction_id|order_id|product_id|quantity|unit_price|discount_amount|pre_discount_value|discount_percentage|
+--------------+--------+----------+--------+----------+---------------+------------------+-------------------+
|T00010291     |O0006752|P00254    |5       |299.98    |224.99         |1499.90           |15.00              |
|T00038332     |O0025926|P00050    |5       |299.81    |224.86         |1499.05           |15.00              |
|T00025790     |O0014011|P00915    |5       |299.80    |224.85         |1499.00           |15.00              |
|T00007387     |O0015728|P00620    |5       |299.73    |224.80         |1498.65           |15.00              |
|T00040454     |O0025052|P00160    |5       |299.72    |224.79         |1498.60           |15.00              |
|T00014436     |O0020138|P00320    |5       |299.71    |224.78         |1498.55           |15.00        

### Tax Analysis

In [30]:
df_tax_analysis = spark.sql("""
    SELECT
        COUNT(DISTINCT transaction_id) AS total_transactions,

        ROUND(SUM(tax_amount), 2) AS total_tax_amount,

        ROUND(AVG(tax_amount), 2) AS average_tax_amount,

        ROUND(MAX(tax_amount), 2) AS maximum_tax_amount,

        COUNT(DISTINCT CASE
            WHEN tax_amount > 0 THEN transaction_id
        END) AS transactions_with_tax,

        COUNT(DISTINCT CASE
            WHEN tax_amount = 0 THEN transaction_id
        END) AS transactions_without_tax,

        COUNT(DISTINCT CASE
            WHEN tax_amount IS NULL THEN transaction_id
        END) AS transactions_with_null_tax,

        ROUND(
            SUM(tax_amount) * 100.0 /
            NULLIF(SUM(transaction_amount), 0),
            2
        ) AS tax_percentage_of_transaction_amount

    FROM gold_fact_sales
""")

df_tax_analysis.show(truncate=False)

+------------------+----------------+------------------+------------------+---------------------+------------------------+--------------------------+------------------------------------+
|total_transactions|total_tax_amount|average_tax_amount|maximum_tax_amount|transactions_with_tax|transactions_without_tax|transactions_with_null_tax|tax_percentage_of_transaction_amount|
+------------------+----------------+------------------+------------------+---------------------+------------------------+--------------------------+------------------------------------+
|49990             |1740878.42      |34.82             |119.94            |49990                |0                       |0                         |7.41                                |
+------------------+----------------+------------------+------------------+---------------------+------------------------+--------------------------+------------------------------------+



## Product Pricing & Profitability Analysis

### Objective
Analyze product pricing and estimated gross profitability by comparing product selling prices with cost prices using the Gold sales fact table and Gold product dimension.

### Business questions
- Which products generate the highest estimated gross profit?
- Which product categories have the highest estimated gross profit margin?

### Key metrics
- Total quantity sold
- Total product cost
- Total estimated gross profit
- Estimated gross profit margin percentage
- Product-level estimated gross profit
- Category-level estimated gross profit

### Product Level Profitability

In [36]:
df_product_profitability = spark.sql("""
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.brand,
        SUM(f.quantity) AS total_quantity_sold,

        ROUND(
            SUM(p.cost_price * f.quantity), 2
        ) AS total_product_cost,

        ROUND(
            SUM(f.unit_price * f.quantity), 2
        ) AS total_pre_discount_value,

        ROUND(
            SUM((f.unit_price - p.cost_price) * f.quantity), 2
        ) AS estimated_gross_profit,

        ROUND(
            SUM((f.unit_price - p.cost_price) * f.quantity) * 100.0 /
            NULLIF(SUM(f.unit_price * f.quantity), 0),
            2
        ) AS estimated_gross_profit_margin_percentage

    FROM gold_fact_sales f
    INNER JOIN gold_dim_product p
        ON f.product_id = p.product_id

    GROUP BY
        p.product_id,
        p.product_name,
        p.category,
        p.brand

    ORDER BY estimated_gross_profit DESC
""")

df_product_profitability.show(truncate=False)

+----------+-----------------------+--------------+------+-------------------+------------------+------------------------+----------------------+----------------------------------------+
|product_id|product_name           |category      |brand |total_quantity_sold|total_product_cost|total_pre_discount_value|estimated_gross_profit|estimated_gross_profit_margin_percentage|
+----------+-----------------------+--------------+------+-------------------+------------------+------------------------+----------------------+----------------------------------------+
|P00997    |Harbor Phones 0997     |Electronics   |Atlas |210                |6541.50           |38598.21                |32056.71              |83.05                                   |
|P00600    |Peak Furniture 0600    |Home & Kitchen|Peak  |201                |3310.47           |34134.64                |30824.17              |90.30                                   |
|P00087    |Vertex Fiction 0087    |Books         |Nova  |196    

### Category Level Profitability

In [35]:
df_category_profitability = spark.sql("""
    SELECT
        COALESCE(p.category, 'UNKNOWN') AS category,
        SUM(f.quantity) AS total_quantity_sold,

        ROUND(
            SUM(p.cost_price * f.quantity), 2
        ) AS total_product_cost,

        ROUND(
            SUM(f.unit_price * f.quantity), 2
        ) AS total_pre_discount_value,

        ROUND(
            SUM((f.unit_price - p.cost_price) * f.quantity), 2
        ) AS estimated_gross_profit,

        ROUND(
            SUM((f.unit_price - p.cost_price) * f.quantity) * 100.0 /
            NULLIF(SUM(f.unit_price * f.quantity), 0),
            2
        ) AS estimated_gross_profit_margin_percentage

    FROM gold_fact_sales f
    INNER JOIN gold_dim_product p
        ON f.product_id = p.product_id

    GROUP BY COALESCE(p.category, 'UNKNOWN')
    ORDER BY estimated_gross_profit DESC
""")

df_category_profitability.show(truncate=False)

+--------------+-------------------+------------------+------------------------+----------------------+----------------------------------------+
|category      |total_quantity_sold|total_product_cost|total_pre_discount_value|estimated_gross_profit|estimated_gross_profit_margin_percentage|
+--------------+-------------------+------------------+------------------------+----------------------+----------------------------------------+
|Books         |29351              |2606052.24        |4497900.83              |1891848.59            |42.06                                   |
|Home & Kitchen|31133              |2892601.30        |4775483.77              |1882882.47            |39.43                                   |
|Beauty        |30220              |2856180.32        |4615581.55              |1759401.23            |38.12                                   |
|Sports        |30553              |2940676.09        |4657283.72              |1716607.63            |36.86                      

## Business KPI Summary

### Objective
Consolidate the most important business performance indicators from the Gold analytics tables to provide a high-level overview of e-commerce business performance.

### Business questions
- What is the overall transaction volume and recorded transaction amount?
- How much discount and tax are recorded, and what is the estimated gross profit?

### Key metrics
- Total transactions
- Total orders
- Total customers
- Total products
- Total quantity sold
- Total transaction amount
- Average transaction amount
- Total discount amount
- Total tax amount
- Estimated gross profit
- Estimated gross profit margin percentage

### Overall Business KPI Dashboard Metrics

In [37]:
df_business_kpi = spark.sql("""
    SELECT
        COUNT(DISTINCT f.transaction_id) AS total_transactions,
        COUNT(DISTINCT f.order_id) AS total_orders,
        COUNT(DISTINCT f.customer_id) AS total_customers,
        COUNT(DISTINCT f.product_id) AS total_products,

        SUM(f.quantity) AS total_quantity_sold,

        ROUND(SUM(f.transaction_amount), 2) AS total_transaction_amount,

        ROUND(
            SUM(f.transaction_amount) /
            NULLIF(COUNT(DISTINCT f.transaction_id), 0),
            2
        ) AS average_transaction_amount,

        ROUND(SUM(f.discount_amount), 2) AS total_discount_amount,

        ROUND(SUM(f.tax_amount), 2) AS total_tax_amount

    FROM gold_fact_sales f
""")

df_business_kpi.show(truncate=False)

+------------------+------------+---------------+--------------+-------------------+------------------------+--------------------------+---------------------+----------------+
|total_transactions|total_orders|total_customers|total_products|total_quantity_sold|total_transaction_amount|average_transaction_amount|total_discount_amount|total_tax_amount|
+------------------+------------+---------------+--------------+-------------------+------------------------+--------------------------+---------------------+----------------+
|49990             |24373       |9109           |1001          |149852             |23501845.78             |470.13                    |1133229.16           |1740878.42      |
+------------------+------------+---------------+--------------+-------------------+------------------------+--------------------------+---------------------+----------------+



### Overall Estimated Gross Profit KPIs

In [38]:
df_profitability_kpi = spark.sql("""
    SELECT
        SUM(f.quantity) AS matched_product_quantity,

        ROUND(
            SUM(p.cost_price * f.quantity), 2
        ) AS total_product_cost,

        ROUND(
            SUM(f.unit_price * f.quantity), 2
        ) AS total_pre_discount_value,

        ROUND(
            SUM((f.unit_price - p.cost_price) * f.quantity), 2
        ) AS estimated_gross_profit,

        ROUND(
            SUM((f.unit_price - p.cost_price) * f.quantity) * 100.0 /
            NULLIF(SUM(f.unit_price * f.quantity), 0),
            2
        ) AS estimated_gross_profit_margin_percentage

    FROM gold_fact_sales f
    INNER JOIN gold_dim_product p
        ON f.product_id = p.product_id
""")

df_profitability_kpi.show(truncate=False)

+------------------------+------------------+------------------------+----------------------+----------------------------------------+
|matched_product_quantity|total_product_cost|total_pre_discount_value|estimated_gross_profit|estimated_gross_profit_margin_percentage|
+------------------------+------------------+------------------------+----------------------+----------------------------------------+
|149299                  |13897648.02       |22809714.10             |8912066.08            |39.07                                   |
+------------------------+------------------+------------------------+----------------------+----------------------------------------+

